# Module 1 — Chroma: our starting point

In this notebook we build the retrieval side of a RAG app with **Chroma**, the way most of us start: a local folder, a collection, `add`, `query`.

We will do it twice:

1. **With Chroma's default embeddings.** We only pass text, and Chroma computes the vectors for us. Convenient, but it hides an important decision.
2. **With our own Titan Text Embeddings V2 vectors** (1024 dimensions, via Amazon Bedrock). This is the version we compare against OpenSearch, because a fair comparison needs **the same vectors in both stores**.

Prerequisites: run from the `notebooks/` folder, with `.env` created from `.env.example` (see the README).

In [1]:
import sys
from pathlib import Path

# Make `src/` importable when the notebook runs from notebooks/.
sys.path.insert(0, str(Path.cwd().parent))

from pprint import pprint

import chromadb

from src.common import (
    CHROMA_COLLECTION,
    CHROMA_PATH,
    TOP_K,
    load_docs,
    load_queries,
    recall_at_k,
)
from src.embeddings import get_embedder

docs = load_docs()
queries = load_queries()
print(f"{len(docs)} docs, {len(queries)} queries")
print("example doc:", docs[0])
print("example query:", queries[0])

51 docs, 15 queries
example doc: {'id': 'kb-001', 'lang': 'en', 'topic': 'vpn', 'text': "NovaVPN error E-1042 means the client certificate has expired. Open the NovaVPN tray icon, choose 'Renew certificate' and sign in again with SSO. Certificates are valid for 90 days."}
example query: {'id': 'q01', 'kind': 'keyword', 'text': 'What does error E-1042 mean?', 'relevant_ids': ['kb-001']}


## Part A — Let Chroma compute the embeddings

`PersistentClient` stores everything in a local folder (`chroma_db/`, ignored by git). We delete the collection first so the notebook can be re-run from scratch.

In [2]:
client = chromadb.PersistentClient(path=str(CHROMA_PATH))

DEFAULT_COLLECTION = "kb_docs_default_ef"
if DEFAULT_COLLECTION in [c.name for c in client.list_collections()]:
    client.delete_collection(DEFAULT_COLLECTION)

# No embedding function given: Chroma falls back to its default one.
default_col = client.create_collection(DEFAULT_COLLECTION)
default_col.add(
    ids=[d["id"] for d in docs],
    documents=[d["text"] for d in docs],
    metadatas=[{"lang": d["lang"], "topic": d["topic"]} for d in docs],
)

print("embedding function:", default_col.configuration["embedding_function"])
stored = default_col.get(ids=[docs[0]["id"]], include=["embeddings"])
print("vector length Chroma computed for", docs[0]["id"], "->", len(stored["embeddings"][0]))

/Users/carlicode/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:   0%|          | 0.00/79.3M [00:00<?, ?iB/s]

/Users/carlicode/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:   0%|          | 17.0k/79.3M [00:00<09:32, 145kiB/s]

/Users/carlicode/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:   0%|          | 68.0k/79.3M [00:00<05:03, 273kiB/s]

/Users/carlicode/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:   0%|          | 170k/79.3M [00:00<02:50, 487kiB/s] 

/Users/carlicode/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:   0%|          | 391k/79.3M [00:00<01:31, 900kiB/s]

...

/Users/carlicode/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:  95%|█████████▌| 75.4M/79.3M [00:06<00:00, 13.5MiB/s]

/Users/carlicode/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:  97%|█████████▋| 77.0M/79.3M [00:06<00:00, 14.3MiB/s]

/Users/carlicode/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz:  99%|█████████▉| 78.4M/79.3M [00:06<00:00, 14.4MiB/s]

/Users/carlicode/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:07<00:00, 11.8MiB/s]

embedding function: <chromadb.api.types.DefaultEmbeddingFunction object at 0x118682120>
vector length Chroma computed for kb-001 -> 384


We never passed a vector, yet every document has one. Chroma's default embedding function is `all-MiniLM-L6-v2` (run locally with ONNX), which produces **384-dimensional** vectors.

That is the hidden decision: the moment we move to another store, that store will not know about this model. If we let each store embed on its own, we would be comparing two different models, not two databases.

We can still query by text, because Chroma embeds the query with the same default function:

In [3]:
r = default_col.query(query_texts=[queries[5]["text"]], n_results=TOP_K)
print("query:", queries[5]["text"])
for doc_id, dist in zip(r["ids"][0], r["distances"][0]):
    print(f"  {doc_id}  distance={dist:.4f}")

query: I forgot my login secret and cannot get back in
  kb-007  distance=0.7416
  kb-008  distance=1.0854
  kb-011  distance=1.2750


## Part B — Bring our own vectors (Titan Text Embeddings V2)

Now we compute the vectors ourselves with `src/embeddings.py` and hand them to Chroma. The same vectors (cached in `data/cache/`) will go to OpenSearch in Module 2.

Two settings matter:

- `embedding_function=None`: Chroma must never compute vectors for this collection. If we forget a vector, we get an error instead of a silent mix of models.
- `"space": "cosine"`: Titan V2 returns normalized vectors, and cosine is the metric we will also use in OpenSearch (`cosinesimil`). Chroma's default is `l2`.

In [4]:
embedder = get_embedder()
print("embedding provider:", embedder.name, "| dimension:", embedder.dimension)

doc_vectors = embedder.embed([d["text"] for d in docs])
print("vectors:", len(doc_vectors), "x", len(doc_vectors[0]))

embedding provider: titan-v2 | dimension: 1024


vectors: 51 x 1024


In [5]:
if CHROMA_COLLECTION in [c.name for c in client.list_collections()]:
    client.delete_collection(CHROMA_COLLECTION)

collection = client.create_collection(
    CHROMA_COLLECTION,
    configuration={"hnsw": {"space": "cosine"}},
    embedding_function=None,
)
collection.add(
    ids=[d["id"] for d in docs],
    documents=[d["text"] for d in docs],
    metadatas=[{"lang": d["lang"], "topic": d["topic"]} for d in docs],
    embeddings=doc_vectors,
)
print(collection.name, "count:", collection.count())
print("hnsw config:", collection.configuration["hnsw"])

kb_docs count: 51
hnsw config: {'space': 'cosine', 'ef_construction': 100, 'ef_search': 100, 'max_neighbors': 16, 'resize_factor': 1.2, 'sync_threshold': 1000}


### Query with our own query vectors

Because the collection has no embedding function, we embed the queries ourselves and use `query_embeddings` instead of `query_texts`. We keep everything in a `results` dict so we can compare with OpenSearch later.

In [6]:
query_vectors = embedder.embed([q["text"] for q in queries])
response = collection.query(query_embeddings=query_vectors, n_results=TOP_K)

results = {}
for q, ids, dists in zip(queries, response["ids"], response["distances"]):
    results[q["id"]] = {
        "kind": q["kind"],
        "text": q["text"],
        "retrieved": ids,
        "distances": [round(d, 4) for d in dists],
        "relevant": q["relevant_ids"],
        "recall@3": recall_at_k(ids, q["relevant_ids"]),
    }

pprint(results, sort_dicts=False, width=110)

{'q01': {'kind': 'keyword',
         'text': 'What does error E-1042 mean?',
         'retrieved': ['kb-001', 'kb-002', 'kb-031'],
         'distances': [0.4946, 0.6543, 0.6694],
         'relevant': ['kb-001'],
         'recall@3': 1.0},
 'q02': {'kind': 'keyword',
         'text': 'PayFlow E-3007',
         'retrieved': ['kb-031', 'kb-033', 'kb-032'],
         'distances': [0.3276, 0.7383, 0.749],
         'relevant': ['kb-031'],
         'recall@3': 1.0},
 'q03': {'kind': 'keyword',
         'text': 'Orbit MDM E-4115',
         'retrieved': ['kb-014', 'kb-017', 'kb-023'],
         'distances': [0.3445, 0.7, 0.8447],
         'relevant': ['kb-014'],
         'recall@3': 1.0},
 'q04': {'kind': 'keyword',
         'text': 'CAB approval for production changes',
         'retrieved': ['kb-050', 'kb-051', 'kb-020'],
         'distances': [0.4784, 0.687, 0.8338],
         'relevant': ['kb-050', 'kb-051'],
         'recall@3': 1.0},
 'q05': {'kind': 'keyword',
         'text': 'Helix E-5003

In [7]:
print(f"embedding provider: {embedder.name}")
for kind in ("keyword", "semantic", "mixed"):
    scores = [r["recall@3"] for r in results.values() if r["kind"] == kind]
    print(f"mean recall@3 [{kind:<8}] = {sum(scores) / len(scores):.2f}  (n={len(scores)})")

embedding provider: titan-v2
mean recall@3 [keyword ] = 1.00  (n=5)
mean recall@3 [semantic] = 1.00  (n=5)
mean recall@3 [mixed   ] = 1.00  (n=5)


> **What you learned**
> - Chroma computes embeddings for us by default (`all-MiniLM-L6-v2`, 384 dims), which is easy but hides the choice of model.
> - A fair comparison between stores needs identical vectors, so we compute them once (Titan V2, 1024 dims) and pass them explicitly.
> - `embedding_function=None` plus `query_embeddings` makes Chroma a pure vector store, the same role OpenSearch will play.
> - The distance metric is part of the setup: we use cosine in both stores.

Next: `02_opensearch_setup.ipynb`.